# Chapter 07 · Mathematics: Freezing as an optimization constraint

[Chapter guide](README.md) · [Problem-to-mathematics guide](../MATHEMATICS.md) · [Course index](../README.md)

**Status: optional reference for later; not yet reviewed.** The programming lessons are already covered. Continue the programming-led mentorship without needing to complete this companion first. Use small numerical explanations when helpful, and revisit formal notation gradually.

Read one section at a time: words → a small example → symbols → code → your explanation. Symbols are shorthand for ideas you already used. You do not need to memorize all the formulas before continuing.

**Prerequisites:** Weighted sums, squared errors, and gradients; we explain the constrained objective.

The runnable examples use only Python and the installed PyTorch. Start in a fresh kernel and run Setup, then proceed in order. Each example keeps its calculations small and supplies its own data. Code cells are saved unexecuted so you can make your own predictions first.

## Contents

1. [1. Fix some unknowns and learn the rest](#constraint)
2. [2. Derive the bias-only minimum loss](#loss-floor)
3. [3. More trainable parameters allow more solutions](#unfreeze)

## Setup

In [ ]:
import torch

<a id="constraint"></a>
## 1. Fix some unknowns and learn the rest


Normally we can choose all three values in $\hat y=w_1x_1+w_2x_2+b$. Freezing weights at $w_1=w_2=1$ changes the problem to choosing only $b$.

$$\min_b L(w_1=1,w_2=1,b)$$

Read this as **find the bias that makes loss smallest while keeping both weights equal to one**. The minimum symbol states our goal; it is not a separate command that instantly finds it. Freezing imposes a constraint on which values the optimizer may change.

For our two inputs `(1, 0)` and `(0, 1)`, both predictions are $1+b$. Targets are `5` and `3`. The model still contains three scalar parameter values, but only the bias is trainable in this exercise.

In code, a frozen tensor with its old gradient cleared can have `.grad is None`. That does not say the mathematical loss has zero sensitivity to that weight; we chose not to compute or use that gradient. Unfreezing permits new gradients, and the optimizer must also include the parameter for its step to update it.

**Connection:** programming Lessons 01–02 and 04. **Explain:** why does replacing a gradient calculation with a fixed parameter differ from changing the parameter value to zero?


**My prediction or explanation:** _write here_

<a id="loss-floor"></a>
## 2. Derive the bias-only minimum loss


Write down the two squared errors explicitly:

$$L(b)=\frac{(1+b-5)^2+(1+b-3)^2}{2}$$

Let $q=b-3$ to make the arithmetic easier. The two errors become $q-1$ and $q+1$. Expanding the squares gives:

$$(q-1)^2+(q+1)^2=(q^2-2q+1)+(q^2+2q+1)=2q^2+2$$

Divide by two and substitute back:

$$L(b)=(b-3)^2+1$$

A square is never negative. The smallest possible value of $(b-3)^2$ is zero, at $b=3$. Thus the minimum loss is $1$, and both predictions are $4$. This proves the observed loss floor for these frozen values and these targets.

The derivative is $g_b=2(b-3)$. At $b=0$, gradient is `-6`, giving the first SGD update to `0.3` at learning rate `0.05`. At $b=3$, gradient is zero even though loss is `1`: the bias has reached its best allowed value, but the two targets still differ.

**Connection:** programming Lesson 03. **Before running:** if bias becomes `4`, the first target is exactly matched; what happens to the other squared error and the mean loss?


**My prediction or explanation:** _write here_

In [ ]:
# Compare the original two-error formula with the simplified expression.
for bias in [0., 1., 2., 3., 4.]:
    predictions = torch.tensor([1. + bias, 1. + bias])
    targets = torch.tensor([5., 3.])
    direct_loss = ((predictions - targets) ** 2).mean().item()
    simplified_loss = (bias - 3.) ** 2 + 1.
    gradient_bias = 2 * (bias - 3.)
    print(f"bias={bias:.1f} | direct loss={direct_loss:.1f} | "
          f"formula loss={simplified_loss:.1f} | bias gradient={gradient_bias:.1f}")

<a id="unfreeze"></a>
## 3. More trainable parameters allow more solutions


Once both weights can change, fitting our two examples gives two equations:

$$w_1+b=5,\qquad w_2+b=3$$

For any chosen bias $b$, the weights $w_1=5-b$ and $w_2=3-b$ fit these two examples exactly. Two observations do not uniquely determine three parameters here.

At $b=3$, a fitting choice is weights `(2, 0)`. At $b=2$, a fitting choice is weights `(3, 1)`, the original toy rule. Both have zero training loss on these two inputs, but input `(0, 0)` gives different predictions, equal to their respective biases. The original rule's target for that input is `2`.

This explains why unfreezing made training loss approach zero without proving recovery of the original rule. Appropriate independent validation examples help assess which predictions generalize.

**Connection:** programming Lessons 04–05. **Before running:** which of the two fitting parameter choices predicts the original target correctly for input `(0, 0)`?


**My prediction or explanation:** _write here_

In [ ]:
# Exact fits on two examples can disagree on a new example.
train_x = torch.tensor([[1., 0.], [0., 1.]])
train_y = torch.tensor([5., 3.])
unseen_x = torch.tensor([0., 0.])
for weights, bias in [([2., 0.], 3.), ([3., 1.], 2.)]:
    weight_vector = torch.tensor(weights)
    train_predictions = train_x @ weight_vector + bias
    train_loss = ((train_predictions - train_y) ** 2).mean().item()
    unseen_prediction = (unseen_x @ weight_vector + bias).item()
    print("Weights:", weights, "| bias:", bias, "| training loss:", train_loss,
          "| prediction for [0, 0]:", unseen_prediction)

**My observation:** _write here_

Distinguish finding the best point within a restricted set of parameter values from fitting all supplied targets, and distinguish fitting those targets from predicting new ones correctly.